# Document store MCP tools

Exercises `search_code`, `describe_code`, `search_research_library` and
`describe_research_library` over the wire, the way a client reaches them.

**The server must be running** — in VSCode: *yada MCP server (http)*, or:

```
./start_mcp_server.sh
```

It serves `http://127.0.0.1:8081/mcp`. Override with `YADA_MCP_URL`.

These tools return **no text blocks** — everything is in `structured_content`,
derived from the pydantic return annotation. `show_structured` prints that payload,
which is what a client actually reasons over.

In [9]:
from utils import (MCP_URL, call_tool, list_mcp_tools, show_structured,
                   show_tool_schema)

MCP_URL

'http://127.0.0.1:8081/mcp'

In [20]:
await list_mcp_tools()

plot: Create a chart from data supplied in the request and return it as an image.
plot_report: Render a stored time series report as a chart and return it as an image.
search_code: Search Troy Stribling's indexed code repositories by meaning and return ranked excerpts with their file paths (10,763 files, 415,006 indexed chunks).
- query: what to look for, in natural language.
- repo: a repository name, or a GitHub account name to search everything under it.
describe_code: What the code repositories contain: accounts, repositories, file types and counts, under any filters given.
search_research_library: Search Troy Stribling's research library -- papers, research notes and blog posts -- by meaning, returning ranked excerpts (64 files, 4,875 indexed chunks).
- query: what to look for, in natural language.
- authors: who wrote it.
describe_research_library: What the research library contains: authors, titles, shelves and counts, under any filters given.


## 1. The published descriptions

What the client sees before it calls anything, and the only thing steering which
tool it picks and what it puts in the arguments. `shelf`'s values and the author
surnames are **generated from the vocabulary at registration** — a hand-written list
goes stale the first time a document is added, and goes stale silently.

In [22]:
await show_tool_schema('search_code')

search_code
  Search Troy Stribling's indexed code repositories by meaning and return ranked excerpts with their file paths (10,763 files, 415,006 indexed chunks).
- query: what to look for, in natural language.
- repo: a repository name, or a GitHub account name to search everything under it. A partial name works. Omit to search every repository. Call describe_code to see what exists.
- ext: a file extension such as '.py', or a language name such as 'Python', which maps to the extensions that language uses. Pass '(no extension)' for files that have none, such as LICENSE and Makefile. Omit to search all types.
A value that matches nothing is reported with what does exist, and no search is run.

  query                string   required
  repo                 any      optional
  ext                  any      optional
  limit                integer  optional, default=10


{'properties': {'query': {'title': 'Query', 'type': 'string'},
  'repo': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Repo'},
  'ext': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Ext'},
  'limit': {'default': 10, 'title': 'Limit', 'type': 'integer'}},
 'required': ['query'],
 'type': 'object',
 'title': 'search_codeArguments'}

In [12]:
await show_tool_schema('describe_code')

describe_code
  What the code repositories contain: accounts, repositories, file types and counts, under any filters given. No file contents.
Facets include 'account/repo', which pairs every repository with the account that holds it -- the only accounts are glyfish and troystribling. Many repositories are forks, so the account here is who holds this copy, not whoever originally published the project; use this facet rather than assuming, since the two differ often.
describe_code(repo='zgomot') answers 'what is this repository written in'. Call it before search_code when a name or language is uncertain.

  repo                 any      optional
  ext                  any      optional


{'properties': {'repo': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Repo'},
  'ext': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Ext'}},
 'type': 'object',
 'title': 'describe_codeArguments'}

In [13]:
await show_tool_schema('search_research_library')

search_research_library
  Search Troy Stribling's research library -- papers, research notes and blog posts -- by meaning, returning ranked excerpts (64 files, 4,875 indexed chunks).
- query: what to look for, in natural language.
- authors: who wrote it. A surname alone is enough and is preferred -- it reaches every spelling of that name and every co-authored paper: Bates, Besag, Burgin, Busemeyer, Chu, Gell-Mann, Ghosh, Goldstein, Green, Hartle, Jaynes, Martinez, Matthaeus, Meissner, Oughton, Peters, Pothos, Richey, Roberts, Sokal, Stribling, Taleb.
- title: a document title, in full or in part. Call describe_research_library to list them.
- shelf: one of jaynes, notes, posts, publications, reading_list.
A value that matches nothing is reported with what does exist, and no search is run.

  query                string   required
  authors              any      optional
  title                any      optional
  shelf                any      optional
  limit                integer  op

{'properties': {'query': {'title': 'Query', 'type': 'string'},
  'authors': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Authors'},
  'title': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Title'},
  'shelf': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Shelf'},
  'limit': {'default': 10, 'title': 'Limit', 'type': 'integer'}},
 'required': ['query'],
 'type': 'object',
 'title': 'search_research_libraryArguments'}

In [14]:
await show_tool_schema('describe_research_library')

describe_research_library
  What the research library contains: authors, titles, shelves and counts, under any filters given. No document contents. describe_research_library(shelf='jaynes') answers 'who wrote the papers on the Jaynes shelf'. Use it to find exact spellings before searching.

  authors              any      optional
  title                any      optional
  shelf                any      optional


{'properties': {'authors': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Authors'},
  'title': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Title'},
  'shelf': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Shelf'}},
 'type': 'object',
 'title': 'describe_research_libraryArguments'}

## 2. What the code store holds

Unfiltered, this is the whole picture: two accounts, 101 repositories, 190 extensions.

Watch the **`account/repo`** facet. A flat facet can say there are two accounts and
101 repositories but not *which holds which*, and a model asked for a pairing it
cannot see will invent one. Asked for a table of account, project and size, Goose
returned correct repository names and chunk counts with an invented account column —
15 of 20 rows wrong, each fork attributed to its upstream owner (`quantopian` for
pyalgotrade, `mementum` for backtrader) rather than the account that holds this copy.

In [16]:
show_structured(await call_tool('describe_code'), facets=15)

status            ok
size              10,763 documents, 415,006 chunks

  ext -- 190 values  (+175 more)
     .ipynb                                                 191,357
     .csv                                                    64,185
     .txt                                                    39,255
     .py                                                     14,780
     .html                                                   14,348
     .yaml                                                   14,291
     .xml                                                    10,541
     .h                                                       7,827
     .m                                                       7,280
     .rb                                                      6,416
     .c                                                       5,808
     .pbxproj                                                 5,614
     .js                                                      4,963
     .erl 

## 3. Filtered describes

`describe_code(repo='zgomot')` answers *what is this repository written in*. The
answer comes from the per-combination rows, which is why it can be filtered at all:
per-field marginals cannot answer a conjunction, and asking Chroma the same question
costs 3.8 s for a filter matching 384 chunks.

In [18]:
show_structured(await call_tool('describe_code', {'repo': 'zgomot'}))

status            ok
filters_requested {'repo': 'zgomot'}
filters_applied   {'repo': 'zgomot'}
filters_resolved  {'repo': 'exact'}
size              78 documents, 384 chunks

  ext -- 6 values
     .rb                                                        284
     .rdoc                                                       66
     (no extension)                                              18
     .gemspec                                                    12
     .gems                                                        2
     .yml                                                         2

  repo -- 1 values
     zgomot                                                     384

  account -- 1 values
     troystribling                                              384

  account/repo -- 1 values
     troystribling/zgomot                                       384


### Two resolutions in one call

`repo='glyfish'` is an **account**, not a repository — the server classifies the name
rather than making the caller do it, and applies `{'account': 'glyfish'}`, one equality
instead of a 22-value `$in`. `ext='Python'` is a **language**, mapped to the extensions
that language uses. Both decisions are reported in `filters_resolved`.

In [9]:
show_structured(await call_tool(
    'describe_code', {'repo': 'glyfish', 'ext': 'Python'}), facets=10)

status            ok
filters_requested {'repo': 'glyfish', 'ext': 'Python'}
filters_applied   {'account': 'glyfish', 'ext': '.py'}
filters_resolved  {'repo': "account 'glyfish' -> 22 repositories", 'ext': "language 'Python' -> .py"}
size              1,199 documents, 12,310 chunks

  ext -- 1 values
     .py                                                     12,310

  repo -- 21 values  (+11 more)
     zipline                                                  4,692
     backtrader                                               2,336
     pyalgotrade                                              1,522
     alef                                                       759
     alpaca                                                     701
     navi                                                       696
     pylivetrader                                               454
     MCMC                                                       372
     PyMC                                             

### A value that matches nothing

Answered from the vocabulary in microseconds, with what *does* exist. No search runs.
This is the case that used to cost 3 s and return a confident wrong answer: Chroma's
`where` is exact equality, so an unresolvable value matches zero rows and raises
nothing, which is indistinguishable from an empty corpus.

In [10]:
show_structured(await call_tool('describe_code', {'repo': 'nonesuch'}))

status            unresolvable_filter
filters_requested {'repo': 'nonesuch'}
filters_resolved  {'repo': 'no match'}
detail            repo='nonesuch' matches no stored value. `available` lists what exists.

  available repo: alef, BAP3, Studium, pyalgotrade, website, alpaca, meida, zipline, cryptocoins, backtrader, libetpan, ejabberd ...


### Partial names and the extension with no name

`zgom` resolves by substring. A blank `ext` means *files that have none* — LICENSE,
Makefile, Rakefile — which is a real category and the opposite of omitting `ext`,
which searches every extension.

In [11]:
show_structured(await call_tool('describe_code', {'repo': 'zgom'}))

status            ok
filters_requested {'repo': 'zgom'}
filters_applied   {'repo': 'zgomot'}
filters_resolved  {'repo': 'substring, 1 of 101'}
size              78 documents, 384 chunks

  ext -- 6 values
     .rb                                                        284
     .rdoc                                                       66
     (no extension)                                              18
     .gemspec                                                    12
     .gems                                                        2
     .yml                                                         2

  repo -- 1 values
     zgomot                                                     384

  account -- 1 values
     troystribling                                              384

  account/repo -- 1 values
     troystribling/zgomot                                       384


In [12]:
show_structured(await call_tool('describe_code', {'ext': '(no extension)'}), facets=8)

status            ok
filters_requested {'ext': '(no extension)'}
filters_applied   {'ext': ''}
filters_resolved  {'ext': 'files with no extension'}
size              639 documents, 1,995 chunks

  ext -- 1 values
     (no extension)                                           1,995

  repo -- 96 values  (+88 more)
     ejabberd                                                   301
     acts_as_aln                                                173
     has_ancestor                                               168
     cryptocoins                                                123
     MailCore                                                    99
     planb_site                                                  99
     iOSPorts                                                    95
     zipline                                                     66

  account -- 2 values
     troystribling                                            1,668
     glyfish                                       

## 4. Searching code

Ranked excerpts with their metadata and scores. `retrieved` is how many candidates
were considered, `best_score` the top relevance before filtering — and `facets` here
describes **the hits that came back**, not the store.

In [13]:
show_structured(await call_tool('search_code', {
    'query': 'How is the MIDI clock implemented?',
    'repo': 'zgomot',
    'limit': 5,
}), hits=3)

status            ok
filters_requested {'repo': 'zgomot'}
filters_applied   {'repo': 'zgomot'}
filters_resolved  {'repo': 'exact'}
retrieved         5 candidates, best score 0.7

  repo -- 1 values
     zgomot                                                       2

  ext -- 1 values
     .rb                                                          2

  1. [0.70] lib/zgomot/ui/windows.rb
     yada://github/troystribling/zgomot/lib/zgomot/ui/windows.rb
     {'repo': 'zgomot', 'ext': '.rb'}
     "input_window.display(Zgomot::Drivers::Mgr.input || 'None')\n      output_window.display(Zgomot::Drivers::Mgr.output)\n      time_signature_window.display(Zgomot::"...

  2. [0.60] lib/zgomot/comp/progression.rb
     yada://github/troystribling/zgomot/lib/zgomot/comp/progression.rb
     {'repo': 'zgomot', 'ext': '.rb'}
     '#.........................................................................................................\n    # midi interface\n    #...........................'...


## 5. The research library

`authors` is where this work started. The field stores the **whole author list** as
one value — `'Troy Stribling, William H. Matthaeus'` — so naming one contributor
matched nothing and returned no error. A surname now resolves to every spelling and
every co-authored paper.

In [14]:
show_structured(await call_tool('describe_research_library'), facets=8)

status            ok
size              64 documents, 4,875 chunks

  shelf -- 5 values
     notes                                                    2,806
     jaynes                                                     737
     reading_list                                               683
     publications                                               466
     posts                                                      183

  title -- 64 values  (+56 more)
     The Volatility Surface                                     334
     Autoregressive Processes                                   278
     Fractional Brownian Motion                                 256
     Discrete Models of Financial Markets                       250
     Stochastic Calculus for Finance                            234
     Monte Carlo Methods in Statistical Mechanics, Foundations and New Algorithms       191
     The Black-Scholes Model                                    186
     Probability Distributions         

In [15]:
show_structured(await call_tool('search_research_library', {
    'query': 'turbulence anisotropy',
    'authors': 'Matthaeus',
    'limit': 5,
}), hits=3)

status            ok
filters_requested {'authors': 'Matthaeus'}
filters_applied   {'authors': {'$in': ['Sanjoy Ghosh, Troy Stribling, W. H. Matthaeus, Melvyn Goldstein', 'Troy Stribling, S. Oughton, W. H. Matthaeus', 'Troy Stribling, W. H. Matthaeus', 'Troy Stribling, William H. Matthaeus', 'William H. Matthaeus, W. Troy Stribling, Daniel Martinez, and Sean Oughton']}}
filters_resolved  {'authors': 'alias, 5 of 19'}
retrieved         5 candidates, best score 0.4

  authors -- 3 values
     Troy Stribling, William H. Matthaeus                         2
     William H. Matthaeus, W. Troy Stribling, Daniel Martinez, and Sean Oughton         1
     Troy Stribling, S. Oughton, W. H. Matthaeus                  1

  title -- 3 values
     Relaxation processes in a low-order three-dimensional magnetohydrodynamics model         2
     Selective Decay and Coherent Vortices in Two-Dimensional Incompressible Turbulence         1
     Magnetic helicity in magnetohydrodynamic turbulence with a mean 

### Scope the negative to the claim

`authors='Einstein'` is answered without searching. The honest statement is *no
document has Einstein as an author* — not *nothing about Einstein*, which is false:
a content search for Einstein returns Brownian motion documents. `available` is the
next step rather than a dead end.

In [16]:
show_structured(await call_tool('search_research_library', {
    'query': 'relativity', 'authors': 'Einstein'}))

status            unresolvable_filter
filters_requested {'authors': 'Einstein'}
filters_resolved  {'authors': 'no match'}
detail            authors='Einstein' matches no stored value, so no search was run. `available` lists what exists.
retrieved         0 candidates, best score None

  available authors: Troy Stribling, E. T. Jaynes, Alan D. Sokal, Troy Stribling, William H. Matthaeus, Troy Stribling, D. A. Roberts, M. L. Goldstein, Troy Stribling, S. Oughton, W. H. Matthaeus, Matthew Richey, Troy Stribling, W. H. Matthaeus, N. N. Taleb, Nassim Nicholas Taleb, Murray Gell-Mann, James B. Hartle, Julian Besag, Peter J. Green ...


In [17]:
show_structured(await call_tool('describe_research_library', {'shelf': 'jaynes'}), facets=8)

status            ok
filters_requested {'shelf': 'jaynes'}
filters_applied   {'shelf': 'jaynes'}
filters_resolved  {'shelf': 'exact'}
size              13 documents, 737 chunks

  shelf -- 1 values
     jaynes                                                     737

  title -- 13 values  (+5 more)
     Marginalization and Prior Probabilities                     88
     On The Rationale of Maximum-Entropy Methods                 82
     Prior Probabilities                                         77
     Predictive Statistical Mechanics                            67
     Prior Information in Inference                              60
     The Minimum Entropy Production Principle                    55
     Macroscopic Prediction                                      53
     Probability as Logic                                        52

  authors -- 1 values
     E. T. Jaynes                                               737


## 6. The raw payload

What a client parses, unrendered — `output_schema` is derived from the return
annotation and `structured_content` is populated by the SDK, so this is the contract.

In [18]:
import json

result = await call_tool('describe_code', {'repo': 'zgomot', 'ext': '.rb'})
print(json.dumps(result.structured_content, indent=2)[:1800])

{
  "status": "ok",
  "request": "",
  "filters_requested": {
    "repo": "zgomot",
    "ext": ".rb"
  },
  "filters_applied": {
    "repo": "zgomot",
    "ext": ".rb"
  },
  "filters_resolved": {
    "repo": "exact",
    "ext": "exact"
  },
  "filters_relaxed": {},
  "available": {},
  "detail": null,
  "facets": {
    "ext": {
      ".rb": 284
    },
    "repo": {
      "zgomot": 284
    },
    "account": {
      "troystribling": 284
    },
    "account/repo": {
      "troystribling/zgomot": 284
    }
  },
  "documents": 67,
  "chunks": 284
}
